In [3]:
import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, precision_recall_curve, balanced_accuracy_score, accuracy_score, ConfusionMatrixDisplay, classification_report
from netcal.metrics import ECE

from glob import glob

In [4]:
model_set = 'keep/concat'
label2id = {
    'Inaccurate': 0,
    'Accurate': 1
}
n_bins = 7

res = []

for each_result in glob('../' + model_set + '/results/*_preds.csv'):
    dat = pd.read_csv(each_result)

    y_true = [label2id[x] for x in dat.target]
    preds = [label2id[x] for x in dat.pred]
    probs = dat.prob

    model = each_result.split('/')[-1].split('_preds.csv')[0].split('_')

    metrics = {
        'HF Org': model[0],
        'Model': model[1:],
        'Balanced Accuracy': balanced_accuracy_score(y_true, preds),
        'Macro F1': f1_score(y_true, preds, average = 'macro'),
        'AUC': roc_auc_score(y_true, probs),
        'ECE': ECE(bins = n_bins).measure(np.array(probs), np.array(y_true))
    }

    res.append(pd.DataFrame(metrics, index = [0]))

out = np.round(pd.concat(res).sort_values(['Balanced Accuracy'], ascending = False).reset_index(drop = True), 3)
print('Model Count:', len(out))
display(out)

if True:
    out.to_csv('../' + model_set + '/results/overall_results.csv', index = False)

Model Count: 28


,HF Org,Model,Balanced Accuracy,Macro F1,AUC,ECE
0,microsoft,deberta-v3-large,0.682,0.684,0.724,0.024
1,albert,albert-xlarge-v2,0.677,0.679,0.716,0.039
2,microsoft,deberta-large,0.675,0.677,0.724,0.032
3,microsoft,deberta-v3-xsmall,0.674,0.675,0.730,0.050
4,google,electra-small-discriminator,0.673,0.674,0.723,0.045
5,google,electra-base-discriminator,0.672,0.673,0.729,0.038
6,xlnet,xlnet-large-cased,0.670,0.671,0.718,0.072
7,google-bert,bert-large-uncased,0.670,0.671,0.719,0.035
8,distilbert,distilbert-base-uncased,0.668,0.669,0.724,0.049
9,microsoft,deberta-v3-small,0.668,0.668,0.728,0.042
